In [1]:
import astropy.units as u
import astropy.coordinates as apycoords
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
hclu = pd.read_csv("/Volumes/travelpassport/litclusterdatabases/HR24/HR24_clusters.csv")
hmem = pd.read_hdf("/Volumes/travelpassport/litclusterdatabases/HR24/HR24_members.h5")


open_cond = (
    (hclu.dist16 < 500) & (hclu.CST > 5) & (hclu.Type == "o") & (hclu.RV.notna())
)
mg_cond = (hclu.dist16 < 500) & (hclu.CST > 10) & (hclu.Type == "m") & (hclu.RV.notna())
cond = open_cond | mg_cond

hclu = hclu.loc[cond].reset_index(drop=True).copy()

hmem = hmem.loc[hmem.Name.isin(hclu.Name)].reset_index(drop=True).copy()

In [3]:
def xyzuvw(ra, dec, distance, pmra, pmdec, radial_velocity):
    """
    Converts ICRS ra (deg), dec (deg), parallax (mas), pmra (mas/yr),
    pmdec (mas/yr), and radial velocity (km/s) to heliocentric
    Cartesian XYZ (pc) and UVW (km/s)

    Args:
        ra (array-like): Right ascension in degrees
        dec (array-like): Declination in degrees
        distance (array-like): Distance in parsecs
        pmra (array-like): Proper motion in right ascension (mas/yr)
        pmdec (array-like): Proper motion in declination (mas/yr)
        radial_velocity (array-like): Radial velocity in km/s

    Returns:
        numpy.ndarray : (n x 6) XYZUVW array
    """

    c = apycoords.SkyCoord(
        ra=ra * u.deg,
        dec=dec * u.deg,
        distance=distance * u.pc,
        pm_ra_cosdec=pmra * u.mas / u.yr,
        pm_dec=pmdec * u.mas / u.yr,
        radial_velocity=radial_velocity * u.km / u.s,
        frame="icrs",
    )

    cg = c.transform_to(apycoords.Galactic())
    cg.representation_type = "cartesian"

    xyz = cg.cartesian.xyz.value.T.reshape(-1, 3)
    uvw = np.vstack([cg.U.value, cg.V.value, cg.W.value]).T
    xyzuvw = np.concatenate([xyz, uvw], axis=1)

    return xyzuvw

In [ ]:
def galcen_cyl(ra, dec, distance, pmra, pmdec, radial_velocity):
    """
    Converts ICRS ra (deg), dec (deg), parallax (mas), pmra (mas/yr),
    pmdec (mas/yr), and radial velocity (km/s) to Galactocentric cylindrical coordinates
    rho (kpc), phi (deg), z (pc), v_rho (km/s), v_phi (km/s), v_z (km/s)

    LEFT-HANDED

    Args:
        ra (array-like): Right ascension in degrees
        dec (array-like): Declination in degrees
        distance (array-like): Distance in parsecs
        pmra (array-like): Proper motion in right ascension (mas/yr)
        pmdec (array-like): Proper motion in declination (mas/yr)
        radial_velocity (array-like): Radial velocity in km/s

    Returns:
        numpy.ndarray: (n x 6) array of galactocentric cylindrical coordinates


    """

    c = apycoords.SkyCoord(
        ra=ra * u.deg,
        dec=dec * u.deg,
        distance=distance * u.pc,
        pm_ra_cosdec=pmra * u.mas / u.yr,
        pm_dec=pmdec * u.mas / u.yr,
        radial_velocity=radial_velocity * u.km / u.s,
        frame="icrs",
    )

    v_sun = apycoords.CartesianDifferential([11.1, 245.0, 7.25] * u.km / u.s)
    gc_frame = apycoords.Galactocentric(
        galcen_distance=8.25 * u.kpc, z_sun=20.8 * u.pc, galcen_v_sun=v_sun
    )

    gc = c.transform_to(gc_frame)
    gc.representation_type = "cylindrical"

    cyl_coord = np.vstack(
        [
            gc.rho.to(u.kpc).value,
            180 - gc.phi.degree, # 180 - PHI ASTROPY CONVENTION
            gc.z.to(u.kpc).value,
            gc.d_rho.to(u.km / u.s).value,
            -(gc.d_phi * gc.rho)
            .to(u.km / u.s, equivalencies=u.dimensionless_angles())
            .value, # FLIPPED SIGN FROM ASTROPY CONVENTION
            gc.d_z.to(u.km / u.s).value,
        ]
    ).T

    return cyl_coord

In [16]:
def skycoord_cylvel_to_dvT(ra, dec, parallax, pmra, pmdec, v_rho, v_phi, v_z):

    v_sun = apycoords.CartesianDifferential([11.1, 245.0, 7.25] * u.km / u.s)

    gc_frame = apycoords.Galactocentric(
        galcen_distance=8.25 * u.kpc, z_sun=20.8 * u.pc, galcen_v_sun=v_sun
    )

    c = apycoords.SkyCoord(
        ra=ra * u.deg,
        dec=dec * u.deg,
        distance=(1000.0 / parallax) * u.pc,
        frame="icrs",
    )

    gc = c.transform_to(gc_frame)

    cyl = gc.represent_as(apycoords.CylindricalRepresentation)
    phi = cyl.phi

    v_rho = np.asarray(v_rho) * u.km / u.s
    v_phi = -np.asarray(v_phi) * u.km / u.s
    v_z = np.asarray(v_z) * u.km / u.s

    vx = v_rho * np.cos(phi) - v_phi * np.sin(phi)
    vy = v_rho * np.sin(phi) + v_phi * np.cos(phi)
    vz = v_z

    gc_vel = apycoords.Galactocentric(
        x=gc.x,
        y=gc.y,
        z=gc.z,
        v_x=vx,
        v_y=vy,
        v_z=vz,
        galcen_distance=8.25 * u.kpc,
        z_sun=20.8 * u.pc,
        galcen_v_sun=v_sun,
    )

    icrs = apycoords.SkyCoord(gc_vel).transform_to(apycoords.ICRS())

    vra = 4.74047 * pmra / parallax
    vdec = 4.74047 * pmdec / parallax

    proj_pmra = icrs.pm_ra_cosdec.to(u.mas / u.yr).value
    proj_pmdec = icrs.pm_dec.to(u.mas / u.yr).value
    proj_vra = (
        (icrs.pm_ra_cosdec * icrs.distance)
        .to(u.km / u.s, equivalencies=u.dimensionless_angles())
        .value
    )
    proj_vdec = (
        (icrs.pm_dec * icrs.distance)
        .to(u.km / u.s, equivalencies=u.dimensionless_angles())
        .value
    )
    proj_vrad = icrs.radial_velocity.to(u.km / u.s).value

    delta_vra = vra - proj_vra
    delta_vdec = vdec - proj_vdec
    delta_vT = np.sqrt(delta_vra**2 + delta_vdec**2)

    return np.vstack(
        [vra, vdec, proj_vra, proj_vdec, proj_vrad, delta_vra, delta_vdec, delta_vT]
    ).T

In [12]:
clu_params = pd.read_csv("../data/clu_params.csv")

In [18]:
foo = skycoord_cylvel_to_dvT(
    clu_params.ra.values,
    clu_params.dec.values,
    clu_params.median_member_parallax.values,
    clu_params.pmra.values,
    clu_params.pmdec.values,
    clu_params.v_rho.values,
    clu_params.v_phi.values,
    clu_params.v_z.values,
)

In [19]:
foo.shape

(288, 8)

In [21]:
import duckdb

In [22]:
df = duckdb.query("""
    SELECT * FROM '/Volumes/travelpassport/tables/allskylitjoin/sky_00000011.parquet'
    LIMIT 1000
""").df()

In [24]:
df.columns

Index(['source_id', 'ra', 'ra_error', 'dec', 'dec_error', 'parallax',
       'parallax_error', 'parallax_over_error', 'pm', 'pmra', 'pmra_error',
       'pmdec', 'pmdec_error', 'ruwe', 'phot_g_mean_flux',
       'phot_g_mean_flux_error', 'phot_g_mean_mag', 'phot_bp_mean_flux',
       'phot_bp_mean_flux_error', 'phot_bp_mean_mag', 'phot_rp_mean_flux',
       'phot_rp_mean_flux_error', 'phot_rp_mean_mag',
       'phot_bp_rp_excess_factor', 'bp_rp', 'bp_g', 'g_rp', 'radial_velocity',
       'radial_velocity_error', 'l', 'b', 'has_xp_continuous', 'has_rvs', 'X',
       'Y', 'Z', 'U', 'V', 'W', '__index_level_0__', 'source_id_1',
       'ocmg_name_1_SPYGLASS', 'ocmg_name_2_SPYGLASS', 'ocmg_name_3_SPYGLASS',
       'mem_prob_1_SPYGLASS', 'mem_prob_2_SPYGLASS', 'mem_prob_3_SPYGLASS',
       'ocmg_name_1_KCS20', 'ocmg_name_2_KCS20', 'ocmg_name_1_CG20',
       'ocmg_name_2_CG20', 'mem_prob_1_CG20', 'mem_prob_2_CG20',
       'ocmg_name_1_HR24', 'ocmg_name_2_HR24', 'ocmg_name_3_HR24',
       'mem